# Análise Exploratória — Contratos PNCP

Este notebook carrega os dados do MongoDB Atlas e realiza análise exploratória dos contratos públicos coletados pelo pipeline ETL.

In [ ]:
import os
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv('MONGODB_URI'))
db = client[os.getenv('MONGODB_DB', 'pncp_etl')]
collection = db[os.getenv('MONGODB_COLLECTION', 'contratos')]

df = pd.DataFrame(list(collection.find({}, {'_id': 0})))
print(f'Total de registros: {len(df)}')
df.head()

## Estatísticas Descritivas

In [ ]:
df[['valorInicial', 'valorGlobal']].describe()

## Contratos por Órgão (Top 10)

In [ ]:
import plotly.express as px

if 'orgaoEntidade' in df.columns:
    df['orgao_nome'] = df['orgaoEntidade'].apply(
        lambda x: x.get('razaoSocial', 'Desconhecido') if isinstance(x, dict) else 'Desconhecido'
    )
    top_orgaos = df['orgao_nome'].value_counts().head(10).reset_index()
    top_orgaos.columns = ['Órgão', 'Quantidade']
    fig = px.bar(top_orgaos, x='Órgão', y='Quantidade', title='Top 10 Órgãos por Número de Contratos')
    fig.show()

## Distribuição de Valores Iniciais

In [ ]:
if 'valorInicial' in df.columns:
    valores = df['valorInicial'].dropna()
    fig = px.histogram(valores, nbins=50, title='Distribuição dos Valores Iniciais dos Contratos', labels={'value': 'Valor Inicial (R$)'})
    fig.show()

## Timeline de Publicações

In [ ]:
if 'dataPublicacaoPncp' in df.columns:
    df['data'] = pd.to_datetime(df['dataPublicacaoPncp'], errors='coerce')
    por_dia = df.groupby(df['data'].dt.date).size().reset_index(name='contratos')
    fig = px.line(por_dia, x='data', y='contratos', title='Contratos Publicados por Dia')
    fig.show()